# Práctica 08. Representación gráfica de datos y cálculo de medidas de tendencia central y dispersión

**Materia:** Probabilidad y Estadística  
**Archivo de datos:** `COVID19MEXICO.csv`  
**Herramientas:** Python, pandas, numpy y matplotlib

> Nota académica: el archivo incluido es un conjunto de datos **sintético** elaborado para fines didácticos. No debe utilizarse como fuente epidemiológica oficial. Su estructura permite practicar carga, limpieza, análisis descriptivo y visualización de datos relacionados con registros tipo COVID-19 en México.


## 1. Objetivo de la práctica

Analizar un conjunto de datos con registros tipo COVID-19 mediante Python, aplicando técnicas de estadística descriptiva para obtener medidas de tendencia central, medidas de dispersión y representaciones gráficas que faciliten la interpretación de la información.


In [ ]:
# Importación de bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración general de visualización en pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Carga del conjunto de datos

El archivo `COVID19MEXICO.csv` debe encontrarse en la misma carpeta que este notebook.


In [ ]:
# Cargar datos
archivo = 'COVID19MEXICO.csv'
df = pd.read_csv(archivo)

# Vista inicial del conjunto de datos
display(df.head())
print('Dimensiones del dataset:', df.shape)


## 3. Revisión de estructura y limpieza básica

Antes de calcular medidas o construir gráficas, es necesario revisar tipos de datos, valores faltantes y consistencia mínima de las columnas.


In [ ]:
# Información general del DataFrame
df.info()

# Conteo de valores faltantes por columna
print('\nValores faltantes por columna:')
display(df.isna().sum())


In [ ]:
# Conversión de fechas y validación de columnas numéricas
columnas_fecha = ['FECHA_ACTUALIZACION', 'FECHA_SINTOMAS', 'FECHA_INGRESO']
for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], errors='coerce')

columnas_numericas = ['EDAD', 'DIAS_SINTOMAS_INGRESO', 'DIAS_ESTANCIA']
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Eliminar duplicados exactos, si existieran
df = df.drop_duplicates()

# Confirmación de tipos de datos después de la limpieza
print(df.dtypes)


## 4. Diccionario breve de variables

| Variable | Descripción |
|---|---|
| `EDAD` | Edad registrada del paciente. |
| `SEXO` | Sexo registrado. |
| `ENTIDAD_RES` | Entidad federativa de residencia. |
| `TIPO_PACIENTE` | Clasificación ambulatoria u hospitalizada. |
| `DIAS_ESTANCIA` | Días de estancia hospitalaria. En pacientes ambulatorios se registra como 0. |
| `RESULTADO_COVID` | Clasificación del resultado: positivo, negativo o sospechoso. |
| `DIABETES`, `HIPERTENSION`, `OBESIDAD`, etc. | Variables categóricas de comorbilidad. |


## 5. Estadística descriptiva inicial

Se obtienen conteos, porcentajes y un resumen estadístico para variables numéricas seleccionadas.


In [ ]:
# Resumen estadístico de variables numéricas
display(df[columnas_numericas].describe())

# Conteo por resultado COVID
print('Distribución de RESULTADO_COVID:')
display(df['RESULTADO_COVID'].value_counts().to_frame('frecuencia'))

# Porcentaje por resultado COVID
print('Porcentaje por RESULTADO_COVID:')
display((df['RESULTADO_COVID'].value_counts(normalize=True) * 100).round(2).to_frame('porcentaje'))


## 6. Medidas de tendencia central para datos no agrupados

Se calculan la media, media recortada, mediana y moda. La media recortada se obtiene eliminando un porcentaje de valores extremos en ambos extremos de la distribución.


In [ ]:
def media_recortada(serie, proporcion=0.10):
    """Calcula la media recortada eliminando una proporción de datos en cada extremo."""
    x = serie.dropna().sort_values().to_numpy()
    k = int(len(x) * proporcion)
    if len(x) == 0 or len(x) <= 2 * k:
        return np.nan
    return np.mean(x[k:len(x)-k])

def moda_unica_o_multiple(serie):
    modas = serie.dropna().mode()
    return ', '.join(map(str, modas.tolist()))

variable = 'EDAD'
medidas_tendencia = pd.DataFrame({
    'Medida': ['Media', 'Media recortada 10%', 'Mediana', 'Moda'],
    'Valor': [
        df[variable].mean(),
        media_recortada(df[variable], 0.10),
        df[variable].median(),
        moda_unica_o_multiple(df[variable])
    ]
})

display(medidas_tendencia)


### Interpretación sugerida

Explique con sus propias palabras qué representa cada medida. Compare la media y la mediana: si ambas son muy distintas, puede existir sesgo o influencia de valores extremos.


## 7. Medidas de dispersión para datos no agrupados

Se calculan rango, varianza, desviación estándar, cuartiles y rango intercuartílico.


In [ ]:
variable = 'EDAD'
q1 = df[variable].quantile(0.25)
q3 = df[variable].quantile(0.75)

medidas_dispersion = pd.DataFrame({
    'Medida': [
        'Mínimo', 'Máximo', 'Rango', 'Varianza muestral',
        'Desviación estándar muestral', 'Q1', 'Q2 / Mediana', 'Q3',
        'Rango intercuartílico'
    ],
    'Valor': [
        df[variable].min(),
        df[variable].max(),
        df[variable].max() - df[variable].min(),
        df[variable].var(ddof=1),
        df[variable].std(ddof=1),
        q1,
        df[variable].median(),
        q3,
        q3 - q1
    ]
})

display(medidas_dispersion)


## 8. Tabla de frecuencias para datos no agrupados

En esta sección se construye una tabla de frecuencias usando la variable `RESULTADO_COVID`.


In [ ]:
tabla_no_agrupada = df['RESULTADO_COVID'].value_counts().rename_axis('Resultado').reset_index(name='Frecuencia absoluta')
tabla_no_agrupada['Frecuencia relativa'] = tabla_no_agrupada['Frecuencia absoluta'] / tabla_no_agrupada['Frecuencia absoluta'].sum()
tabla_no_agrupada['Frecuencia acumulada'] = tabla_no_agrupada['Frecuencia absoluta'].cumsum()
tabla_no_agrupada['Frecuencia relativa acumulada'] = tabla_no_agrupada['Frecuencia relativa'].cumsum()

display(tabla_no_agrupada)


## 9. Tabla de frecuencias para datos agrupados

Se agrupa la variable `EDAD` en intervalos de 10 años. También se calcula la marca de clase y una aproximación de la media y varianza para datos agrupados.


In [ ]:
# Intervalos para edad
bins = list(range(0, 111, 10))
intervalos = pd.cut(df['EDAD'], bins=bins, right=False, include_lowest=True)

tabla_agrupada = intervalos.value_counts().sort_index().rename_axis('Intervalo').reset_index(name='Frecuencia absoluta')
tabla_agrupada['Marca de clase'] = tabla_agrupada['Intervalo'].astype(object).apply(lambda x: float((x.left + x.right) / 2))
tabla_agrupada['Frecuencia relativa'] = tabla_agrupada['Frecuencia absoluta'] / tabla_agrupada['Frecuencia absoluta'].sum()
tabla_agrupada['Frecuencia acumulada'] = tabla_agrupada['Frecuencia absoluta'].cumsum()
tabla_agrupada['Frecuencia relativa acumulada'] = tabla_agrupada['Frecuencia relativa'].cumsum()

display(tabla_agrupada)

# Medidas aproximadas para datos agrupados
n = tabla_agrupada['Frecuencia absoluta'].sum()
media_agrupada = (tabla_agrupada['Marca de clase'] * tabla_agrupada['Frecuencia absoluta']).sum() / n
varianza_agrupada = (tabla_agrupada['Frecuencia absoluta'] * (tabla_agrupada['Marca de clase'] - media_agrupada)**2).sum() / (n - 1)

print(f'Media aproximada para datos agrupados: {media_agrupada:.2f}')
print(f'Varianza aproximada para datos agrupados: {varianza_agrupada:.2f}')
print(f'Desviación estándar aproximada para datos agrupados: {np.sqrt(varianza_agrupada):.2f}')


## 10. Histograma

El histograma permite observar la forma general de la distribución de una variable cuantitativa.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['EDAD'].dropna(), bins=10, edgecolor='black')
plt.title('Histograma de edades')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.grid(axis='y', alpha=0.3)
plt.show()


**Pregunta de análisis:** ¿La distribución de la edad parece simétrica, sesgada a la derecha o sesgada a la izquierda? Justifique su respuesta.


## 11. Gráfica de dispersión

La gráfica de dispersión permite observar la relación visual entre dos variables cuantitativas. En este caso se compara la edad con los días de estancia.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['EDAD'], df['DIAS_ESTANCIA'], alpha=0.6)
plt.title('Edad vs. días de estancia')
plt.xlabel('Edad')
plt.ylabel('Días de estancia')
plt.grid(alpha=0.3)
plt.show()


**Pregunta de análisis:** ¿Se observa alguna tendencia entre edad y días de estancia? ¿Existen valores que podrían considerarse atípicos?


## 12. Diagrama de caja y bigotes

El diagrama de caja y bigotes permite identificar mediana, dispersión, rango intercuartílico y posibles valores atípicos.


In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot(df['EDAD'].dropna(), vert=True, labels=['EDAD'])
plt.title('Diagrama de caja y bigotes de la edad')
plt.ylabel('Edad')
plt.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:
# Identificación de valores atípicos usando el criterio 1.5 * IQR
q1 = df['EDAD'].quantile(0.25)
q3 = df['EDAD'].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

atipicos = df[(df['EDAD'] < limite_inferior) | (df['EDAD'] > limite_superior)]
print('Límite inferior:', limite_inferior)
print('Límite superior:', limite_superior)
print('Número de posibles valores atípicos en EDAD:', len(atipicos))
display(atipicos[['ID_REGISTRO', 'ENTIDAD_RES', 'SEXO', 'EDAD', 'TIPO_PACIENTE', 'RESULTADO_COVID']].head(10))


## 13. Gráfica de tallo y hojas

La gráfica de tallo y hojas conserva los valores originales y permite observar su distribución. En este ejemplo, el tallo representa las decenas de edad y las hojas representan las unidades.


In [ ]:
def tallo_hojas(valores):
    valores = sorted([int(v) for v in valores.dropna()])
    tallos = {}
    for v in valores:
        tallo = v // 10
        hoja = v % 10
        tallos.setdefault(tallo, []).append(hoja)
    for tallo, hojas in tallos.items():
        hojas_txt = ' '.join(str(h) for h in hojas)
        print(f'{tallo:>2} | {hojas_txt}')

print('Tallo | Hojas para EDAD')
tallo_hojas(df['EDAD'])


## 14. Diagrama de Pareto

El diagrama de Pareto permite ordenar categorías de mayor a menor frecuencia y observar el porcentaje acumulado. Aquí se analizan comorbilidades reportadas con valor `SI`.


In [ ]:
comorbilidades = ['NEUMONIA', 'DIABETES', 'EPOC', 'ASMA', 'HIPERTENSION', 'OBESIDAD', 'TABAQUISMO']
conteos = pd.Series({col: (df[col] == 'SI').sum() for col in comorbilidades}).sort_values(ascending=False)
pareto = conteos.reset_index()
pareto.columns = ['Comorbilidad', 'Frecuencia']
pareto['Porcentaje acumulado'] = pareto['Frecuencia'].cumsum() / pareto['Frecuencia'].sum() * 100

display(pareto)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(pareto['Comorbilidad'], pareto['Frecuencia'])
ax1.set_xlabel('Comorbilidad')
ax1.set_ylabel('Frecuencia')
ax1.set_title('Diagrama de Pareto de comorbilidades')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(pareto['Comorbilidad'], pareto['Porcentaje acumulado'], marker='o')
ax2.set_ylabel('Porcentaje acumulado')
ax2.set_ylim(0, 110)
ax2.grid(False)
plt.show()


**Pregunta de análisis:** ¿Qué comorbilidad debería recibir mayor atención de acuerdo con la frecuencia observada? ¿Por qué el Pareto puede ser más informativo que una tabla simple?


## 15. Gráfico circular

El gráfico circular se utiliza para mostrar proporciones. En este caso se representa la distribución de `RESULTADO_COVID`.


In [ ]:
conteo_resultado = df['RESULTADO_COVID'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(conteo_resultado, labels=conteo_resultado.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribución porcentual de RESULTADO_COVID')
plt.show()


## 16. Gráfica de barras por entidad federativa

Esta gráfica permite comparar la cantidad de registros por entidad de residencia.


In [ ]:
casos_entidad = df['ENTIDAD_RES'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
casos_entidad.plot(kind='bar')
plt.title('Registros por entidad federativa')
plt.xlabel('Entidad federativa')
plt.ylabel('Número de registros')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.show()


## 17. Exportación de resultados

Se guardan algunas tablas de resultados en archivos CSV para integrarlas al reporte final.


In [ ]:
tabla_no_agrupada.to_csv('tabla_no_agrupada_resultado_covid.csv', index=False, encoding='utf-8-sig')
tabla_agrupada.to_csv('tabla_agrupada_edad.csv', index=False, encoding='utf-8-sig')
medidas_tendencia.to_csv('medidas_tendencia_edad.csv', index=False, encoding='utf-8-sig')
medidas_dispersion.to_csv('medidas_dispersion_edad.csv', index=False, encoding='utf-8-sig')
pareto.to_csv('pareto_comorbilidades.csv', index=False, encoding='utf-8-sig')

print('Archivos de resultados generados correctamente.')


## 18. Preguntas para el reporte final

Responda formalmente:

1. ¿Qué variable cuantitativa analizó principalmente y por qué?
2. ¿Cuál es la diferencia entre media, mediana y moda en el contexto de este conjunto de datos?
3. ¿La media y la mediana de la edad son similares? ¿Qué indica esto sobre la distribución?
4. ¿Qué información aporta la desviación estándar?
5. ¿Qué utilidad tiene el diagrama de caja y bigotes para detectar valores atípicos?
6. ¿Qué ventaja ofrece el diagrama de Pareto frente a una gráfica de barras común?
7. ¿Qué gráfica considera más adecuada para comunicar resultados en un informe técnico? Justifique.
8. ¿Qué limitaciones tiene trabajar con un conjunto de datos sintético?
